In [ ]:
import concurrent.futures
import json
import os
import re
import shutil
import sqlite3
import threading
import time
import urllib.request

DB_PATH = "weblog_corpus.db"
BACKUP_PATH = "weblog_corpus.db.before_llm_clean.bak"
AUDIT_LOG = "llm_clean_audit.jsonl"
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:7b"
MAX_WORKERS = 3

# 触发大模型检查的“疑似污染”信号（宽松触发，宁可多查几百篇也不漏网）
SUSPECT_RE = re.compile(
    r"[\x00-\x08\x0b\x0c\x0e-\x1f\ufffd]|"
    r"Tranfer path:|Transfer path:|Subject:\s*Homework|"
    r"[^\w\s.,;:'\"!?()\[\]{}\-–—/\\@#$%&*+=<>|~`^‘’“”…£€¥¢§°±×÷]{3,}"
)

SYSTEM_PROMPT = """You are a strict archival text restorer for economist J. Bradford DeLong's blog.
Your job is to clean corrupted text while preserving 100% of authentic human-written prose VERBATIM.

RULES:
1. If the entire input is unreadable binary garbage, random symbols, or spam/email routing headers with no real blog prose, output EXACTLY:
   {"status": "PURE_GARBAGE", "clean_text": ""}
2. If the input contains real English/Chinese blog writing mixed with garbled encoding symbols, binary chunks, or spam headers:
   - Remove ONLY the garbled binary/symbol sequences and spam routing headers.
   - Fix obvious broken encoding characters (like stray replacement boxes) if the surrounding word is clear.
   - Keep ALL legitimate paragraphs, quotes, and sentences WORD-FOR-WORD. Do NOT summarize, paraphrase, or shorten valid prose.
   - Output JSON: {"status": "REPAIRED", "clean_text": "<the verbatim cleaned text>"}

Respond ONLY with valid JSON."""


def is_suspect_doc(body: str) -> bool:
    if not body or len(body.strip()) < 25:
        return True
    if SUSPECT_RE.search(body):
        return True
    # 检查是否存在异常超长无空格乱码词
    no_url = re.sub(r"https?://\S+", "", body)
    words = no_url.split()
    if words and max(len(w) for w in words) > 55:
        return True
    return False


def clean_chunk_with_ollama(text_chunk: str):
    payload = {
        "model": MODEL_NAME,
        "system": SYSTEM_PROMPT,
        "prompt": f"Clean the following archival text:\n\n{text_chunk}",
        "format": "json",
        "stream": False,
        "keep_alive": "20m",
        "options": {
            "temperature": 0.0,
            "num_ctx": 8192,
            "num_predict": 4096,
        },
    }
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        res = json.loads(resp.read().decode("utf-8"))
        parsed = json.loads(res.get("response", "{}"))
        status = parsed.get("status", "")
        clean_text = str(parsed.get("clean_text", "")).strip()
        return status, clean_text


def process_record(rec):
    pid, title, rel_path, body = rec
    # 若文章极长，仅对包含疑似乱码的段落窗口送入模型，避免截断或过度耗时
    paragraphs = re.split(r"\n\s*\n", body)
    if len(body) > 6000 and len(paragraphs) > 1:
        rebuilt = []
        any_repaired = False
        for p in paragraphs:
            if is_suspect_doc(p):
                st, cleaned = clean_chunk_with_ollama(p[:5500])
                if st == "REPAIRED" and cleaned:
                    rebuilt.append(cleaned)
                    any_repaired = True
                elif st == "PURE_GARBAGE":
                    any_repaired = True
                else:
                    rebuilt.append(p)
            else:
                rebuilt.append(p)
        final_text = "\n\n".join(rebuilt).strip()
        if len(final_text.split()) < 25:
            return pid, rel_path, "PURE_GARBAGE", body, ""
        return pid, rel_path, ("REPAIRED" if any_repaired else "UNCHANGED"), body, final_text

    # 常规长度文档整篇送入模型精确修复
    st, cleaned = clean_chunk_with_ollama(body[:6000])
    if st == "PURE_GARBAGE" or len(cleaned.split()) < 25:
        return pid, rel_path, "PURE_GARBAGE", body, ""
    if st == "REPAIRED" and cleaned:
        return pid, rel_path, "REPAIRED", body, cleaned
    return pid, rel_path, "UNCHANGED", body, body


def main():
    print(f"1/4 正在创建安全备份: {BACKUP_PATH} ...", flush=True)
    shutil.copy2(DB_PATH, BACKUP_PATH)

    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    rows = cur.execute("SELECT id, title, rel_path, body_clean FROM posts").fetchall()

    suspects = [r for r in rows if is_suspect_doc(r[3] or "")]
    print(f"2/4 全库 {len(rows):,} 条记录中，筛出疑似含乱码/污染样本 {len(suspects):,} 条，开始调用 {MODEL_NAME} 清洗...", flush=True)

    repaired_updates = []
    garbage_ids = []
    t0 = time.time()
    done_cnt = 0
    lock = threading.Lock()

    with open(AUDIT_LOG, "w", encoding="utf-8") as log_f:
        with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(process_record, r) for r in suspects]
            for fut in concurrent.futures.as_completed(futures):
                try:
                    pid, rel_path, status, orig_body, new_body = fut.result()
                    with lock:
                        done_cnt += 1
                        if status == "PURE_GARBAGE":
                            garbage_ids.append((pid,))
                            log_f.write(json.dumps({
                                "id": pid, "rel_path": rel_path, "action": "ISOLATED_GARBAGE",
                                "sample": orig_body[:250]
                            }, ensure_ascii=False) + "\n")
                        elif status == "REPAIRED" and new_body != orig_body:
                            repaired_updates.append((new_body, pid))
                            log_f.write(json.dumps({
                                "id": pid, "rel_path": rel_path, "action": "REPAIRED_TEXT",
                                "before_sample": orig_body[:250],
                                "after_sample": new_body[:250]
                            }, ensure_ascii=False) + "\n")

                        if done_cnt % 25 == 0 or done_cnt == len(suspects):
                            log_f.flush()
                            rate = done_cnt / max(time.time() - t0, 0.1)
                            eta = (len(suspects) - done_cnt) / max(rate, 0.1)
                            print(f"  进度: {done_cnt}/{len(suspects)} | 速度: {rate:.1f} 条/秒 | 预计剩余: {eta:.0f}s", flush=True)
                except Exception as e:
                    with lock:
                        done_cnt += 1
                        print(f"  [跳过异常] {e}", flush=True)

    print("3/4 正在写回修复结果并重建 FTS5 索引...", flush=True)
    if repaired_updates:
        cur.executemany("UPDATE posts SET body_clean = ? WHERE id = ?", repaired_updates)
    if garbage_ids:
        cur.executemany(
            "UPDATE posts SET is_unique_corpus = 0, is_canonical_post = 0, post_genre = 'corrupt_binary' WHERE id = ?",
            garbage_ids,
        )
    cur.execute("INSERT INTO posts_fts(posts_fts) VALUES('rebuild');")
    conn.commit()
    conn.close()

    print(f"\n4/4 清洗完成！")
    print(f"  - LLM 成功剥离乱码并挽救正文: {len(repaired_updates):,} 篇")
    print(f"  - 确认为纯二进制乱码并隔离  : {len(garbage_ids):,} 篇")
    print(f"  - 前后对照审计日志已保存至  : {AUDIT_LOG}")


if __name__ == "__main__":
    main()